# Dice.com Job Postings — Unique Job Profile Extraction

Loads `dice_com-job_us_sample.csv` and reduces it to one representative row per unique job title, as a candidate replacement data source for `job-profile/*.txt`.

In [1]:
import pandas as pd

df = pd.read_csv("dice_com-job_us_sample.csv")
df.shape

(22000, 12)

## Data overview

In [2]:
df.dtypes

advertiserurl               str
company                     str
employmenttype_jobstatus    str
jobdescription              str
jobid                       str
joblocation_address         str
jobtitle                    str
postdate                    str
shift                       str
site_name                   str
skills                      str
uniq_id                     str
dtype: object

In [3]:
df.isna().sum()

advertiserurl                   0
company                        50
employmenttype_jobstatus      230
jobdescription                  0
jobid                           0
joblocation_address             3
jobtitle                        0
postdate                        0
shift                         357
site_name                   18510
skills                         43
uniq_id                         0
dtype: int64

## Normalize job title

Strips whitespace and stray surrounding quote characters present in some `jobtitle` values, then lowercases for grouping.

In [4]:
df["job_profile"] = (
    df["jobtitle"]
    .str.strip()
    .str.strip('"')
    .str.strip()
    .str.lower()
)
df["job_profile"].nunique(), df.shape[0]

(14924, 22000)

## Keep one row per unique job profile

For each `job_profile`, keeps the row whose `jobdescription` is longest — the most complete posting text to carry forward.

In [5]:
longest_idx = df.groupby("job_profile")["jobdescription"].apply(lambda s: s.str.len().idxmax())
df_unique = df.loc[longest_idx.values].reset_index(drop=True)
df_unique.shape

(14924, 13)

In [6]:
df_unique[["jobtitle", "jobdescription"]].head(10)

,jobtitle,jobdescription
0,# Lead Windows Mobile Developer with BLE or Bl...,Lead Windows Mobile Developer with BLE or Bl...
1,# Sr/Jr Project Manager (Physical Security) / ...,Position Title: Senior / Junior Physical Secur...
2,(2) VOIP Engineers,"Network Engineer (50% support, 50% project wor..."
3,(4223) Build Release Engineer - Redmond WA - O...,"Job Category:Core EngineeringLocationRedmond,..."
4,"(8034) Network Engineer, Deployment and Support","Job Title:Network Engineer, Deployment and Sup..."
5,"(FULL TIME) .NET Developer - Los Angeles, CA","Net2Source, Inc. is one of the fastest growing..."
6,(FULL TIME) Build and Release Engineer - Los A...,"Net2Source, Inc. is one of the fastest growing..."
7,"(FULL TIME) Front End Developer - Texas, North...","Net2Source, Inc. is one of the fastest growing..."
8,"(FULL TIME) HP Exstream Consultant - Plano, TX","Net2Source, Inc. is one of the fastest growing..."
9,"(Full Time) JAVA Developer @ NYC, NY",Position: Senior Java Developer Loca...


## Save result

In [7]:
df_unique[["jobtitle", "jobdescription"]].to_csv("dice_com_unique_job_profiles.csv", index=False)

## Match real postings to the current 10 job profiles

Replaces the artificial `job-profile/*.txt` postings with real Dice.com postings, keeping the same 10 roles. Each profile is matched by a keyword pattern in `jobtitle`; the match with the longest `jobdescription` is kept as the most complete real posting for that role.

In [8]:
profile_patterns = {
    "backend-engineer": r"back[\s-]?end",
    "cloud-engineer": r"cloud",
    "data-engineer": r"data\s+engineer",
    "data-scientist-engineer": r"data\s+scientist",
    "devops-engineer": r"dev\s?ops",
    "frontend-engineer": r"front[\s-]?end",
    "full-stack-engineer": r"full[\s-]?stack",
    "it-security-engineer": r"security\s+engineer",
    "java-developer": r"java\s+developer",
    "ui-ux-designer": r"ui\s*/?\s*ux\s+designer|ux\s+designer|ui\s+designer|user\s+experience\s+designer",
}

match_counts = {
    profile: df_unique["jobtitle"].str.contains(pattern, case=False, regex=True, na=False).sum()
    for profile, pattern in profile_patterns.items()
}
pd.Series(match_counts, name="matches")

backend-engineer           103
cloud-engineer             267
data-engineer               65
data-scientist-engineer     40
devops-engineer            209
frontend-engineer          208
full-stack-engineer        209
it-security-engineer       114
java-developer             348
ui-ux-designer              61
Name: matches, dtype: int64

In [9]:
rows = []
for profile, pattern in profile_patterns.items():
    matched = df_unique[df_unique["jobtitle"].str.contains(pattern, case=False, regex=True, na=False)]
    best = matched.loc[matched["jobdescription"].str.len().idxmax()]
    rows.append({"job_profile": profile, "jobtitle": best["jobtitle"], "jobdescription": best["jobdescription"]})

df_ten = pd.DataFrame(rows)
df_ten.shape

(10, 3)

In [10]:
df_ten[["job_profile", "jobtitle", "jobdescription"]]

,job_profile,jobtitle,jobdescription
0,backend-engineer,"Backend Technical Consultant, Professional Ser...",Build software. Reach 500 million users. Chang...
1,cloud-engineer,CLOUD & INFRASTRUCTURE ENGINEER,The Cloud and Infrastructure Engineer will be ...
2,data-engineer,Encompass Hadoop Data Engineer,Where good people build rewarding careers.Thin...
3,data-scientist-engineer,"Data Scientist (Neural Networks/Decision Tree,...",Overview:The Data Scientist develops and execu...
4,devops-engineer,Sr. DevOps Engineer (Windows),Job Position: Sr. DevOps Engineer (Windows)Loc...
5,frontend-engineer,Front End Designer/Developer,The designer needs Sketch experience! And html...
6,full-stack-engineer,Full Stack Software Engineer,Job Title: Full Stack Software Engineer (17253...
7,it-security-engineer,Security Engineer - CA Siteminder AND Identity...,Position: Security EngineerLocation: Alpharett...
8,java-developer,Java Developers,"Dear Partner,Hope you are doing great today!!P..."
9,ui-ux-designer,Sr. UX Designer,Listen up! There's a revolution afoot in enter...


## Save the 10 matched profiles

In [11]:
df_ten[["job_profile", "jobtitle", "jobdescription"]].to_csv("dice_com_10_job_profiles.csv", index=False)

## Replace the artificial `job-profile/*.txt` postings

Overwrites each existing synthetic posting with the matched real Dice.com posting (title on the first line, description below — same format `ingestion/jp.py` already expects).

In [12]:
from pathlib import Path

profile_to_filename = {
    "backend-engineer": "Backend-Engineer.txt",
    "cloud-engineer": "Cloud-Engineer.txt",
    "data-engineer": "Data-Engineer.txt",
    "data-scientist-engineer": "Data-Scientist-Engineer.txt",
    "devops-engineer": "DevOps-Engineer.txt",
    "frontend-engineer": "Frontend-Engineer.txt",
    "full-stack-engineer": "Full-Stack-Engineer.txt",
    "it-security-engineer": "IT-Security-Engineer.txt",
    "java-developer": "Java Developer.txt",
    "ui-ux-designer": "UI-UX-Designer.txt",
}

for _, row in df_ten.iterrows():
    filename = profile_to_filename[row["job_profile"]]
    title = row["jobtitle"].strip()
    description = row["jobdescription"].replace("\xa0", " ").strip()
    Path(filename).write_text(f"{title}\n\n{description}\n", encoding="utf-8")

list(profile_to_filename.values())

['Backend-Engineer.txt', 'Cloud-Engineer.txt', 'Data-Engineer.txt', 'Data-Scientist-Engineer.txt', 'DevOps-Engineer.txt', 'Frontend-Engineer.txt', 'Full-Stack-Engineer.txt', 'IT-Security-Engineer.txt', 'Java Developer.txt', 'UI-UX-Designer.txt']

## Switch source to `clean_jobs.csv`

More recent postings (2025, LinkedIn/Indeed) than `dice_com-job_us_sample.csv` (2015-2016), but skewed toward data/ML roles: `backend-engineer`, `it-security-engineer`, `java-developer`, and `ui-ux-designer` have zero title matches here. Those 4 profiles are replaced with roles this dataset actually covers — `data-analyst`, `machine-learning-engineer`, `business-intelligence-analyst`, `ai-automation-specialist` — and the other 6 profiles (`data-engineer`, `data-scientist-engineer`, `devops-engineer`, `cloud-engineer`, `frontend-engineer`, `full-stack-engineer`) get their posting re-matched from this dataset too.

In [13]:
df2 = pd.read_csv("clean_jobs.csv").dropna(subset=["title", "description"])
df2.shape

(1044, 10)

In [14]:
df2["job_profile"] = df2["title"].str.strip().str.lower()

longest_idx2 = df2.groupby("job_profile")["description"].apply(lambda s: s.str.len().idxmax())
df2_unique = df2.loc[longest_idx2.values].reset_index(drop=True)
df2_unique.shape

(514, 11)

In [15]:
import re

english_stopwords = {"the","and","to","of","in","for","is","on","with","as","at","by","an","be","this","that","from","or","are","will","you","we","our"}

def english_score(text):
    words = re.findall(r"[a-zA-Z']+", text.lower())
    if not words:
        return 0.0
    sample = words[:400]
    return sum(1 for w in sample if w in english_stopwords) / len(sample)

df2_unique["eng_score"] = df2_unique["description"].apply(english_score)
(df2_unique["eng_score"] < 0.15).sum()

np.int64(57)

In [16]:
profile_patterns_v2 = {
    "data-engineer": r"data\s+engineer",
    "data-scientist-engineer": r"data\s+scientist",
    "devops-engineer": r"dev\s?ops",
    "cloud-engineer": r"cloud",
    "frontend-engineer": r"front[\s-]?end",
    "full-stack-engineer": r"full[\s-]?stack",
    "data-analyst": r"data\s+analyst",
    "machine-learning-engineer": r"machine\s+learning\s+engineer|ai\s*/\s*ml\s+engineer",
    "business-intelligence-analyst": r"business\s+intelligence\s+analyst|analytics\s+engineer|business\s+data\s+analyst",
    "ai-automation-specialist": r"ai\s+automation\s+specialist",
}

rows2 = []
for profile, pattern in profile_patterns_v2.items():
    matched = df2_unique[df2_unique["title"].str.contains(pattern, case=False, regex=True, na=False)]
    english_matched = matched[matched["eng_score"] >= 0.15]
    pool = english_matched if len(english_matched) else matched
    best = pool.loc[pool["description"].str.len().idxmax()]
    rows2.append({"job_profile": profile, "title": best["title"], "description": best["description"]})

df_ten2 = pd.DataFrame(rows2)
df_ten2[["job_profile", "title", "description"]]

,job_profile,title,description
0,data-engineer,Data Engineer III,**About Us**\n\n\n**Kitsap Credit Union** is a...
1,data-scientist-engineer,DE&S Data Scientist,"MOD Abbey Wood, Filton, Bristol, Avon BS34 8JH..."
2,devops-engineer,DevOps Engineer - $20K Sign-On Bonus - 23302,"Enlighten, honored as a Top Workplace from USA..."
3,cloud-engineer,Senior Applications Analyst - Oracle HCM Cloud,**JOB DESCRIPTION** \n\n \n\nThe Senior Appl...
4,frontend-engineer,Junior Frontend Developer,"Department: Enabling Technology, Digital & Tec..."
5,full-stack-engineer,Entry Level Full Stack AI Engineer,**Role Overview**\n=================\n\n\n\nWe...
6,data-analyst,Senior Data Analyst,**Job Information**\n Job Title: Senior Data A...
7,machine-learning-engineer,AI/ML Engineer (early-career),Basic Qualifications\n\nBachelor’s Degree in E...
8,business-intelligence-analyst,Analytics Engineer,About The Team\n\nCome help us build the world...
9,ai-automation-specialist,AI Automation Specialist,"At Foley, we are revolutionizing the way compa..."


## Write the final 10 job profiles

Removes the 4 superseded artificial-profile files and (re)writes all 10 `job-profile/*.txt` files from `clean_jobs.csv`.

In [17]:
for name in ["Backend-Engineer.txt", "IT-Security-Engineer.txt", "Java Developer.txt", "UI-UX-Designer.txt"]:
    Path(name).unlink(missing_ok=True)

profile_to_filename_v2 = {
    "data-engineer": "Data-Engineer.txt",
    "data-scientist-engineer": "Data-Scientist-Engineer.txt",
    "devops-engineer": "DevOps-Engineer.txt",
    "cloud-engineer": "Cloud-Engineer.txt",
    "frontend-engineer": "Frontend-Engineer.txt",
    "full-stack-engineer": "Full-Stack-Engineer.txt",
    "data-analyst": "Data-Analyst.txt",
    "machine-learning-engineer": "Machine-Learning-Engineer.txt",
    "business-intelligence-analyst": "Business-Intelligence-Analyst.txt",
    "ai-automation-specialist": "AI-Automation-Specialist.txt",
}

for _, row in df_ten2.iterrows():
    filename = profile_to_filename_v2[row["job_profile"]]
    title = row["title"].strip()
    description = row["description"].replace("\xa0", " ").strip()
    Path(filename).write_text(f"{title}\n\n{description}\n", encoding="utf-8")

sorted(profile_to_filename_v2.values())

['AI-Automation-Specialist.txt', 'Business-Intelligence-Analyst.txt', 'Cloud-Engineer.txt', 'Data-Analyst.txt', 'Data-Engineer.txt', 'Data-Scientist-Engineer.txt', 'DevOps-Engineer.txt', 'Frontend-Engineer.txt', 'Full-Stack-Engineer.txt', 'Machine-Learning-Engineer.txt']

In [18]:
df_ten2[["job_profile", "title", "description"]].to_csv("clean_jobs_10_job_profiles.csv", index=False)